In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.drought_indices import calculate_spi, calculate_spei, extract_drought_events
from src.drought_probability import calculate_return_periods

print("✅ Setup complete.")

c:\Users\User\miniconda3\envs\hydro-ml\Lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.11.17) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ Setup complete.


In [2]:
# 1. Load the main dataset (which already contains 'pet')
parquet_path = PROJECT_ROOT / "outputs" / "processed" / "sarawak_ML_corrected.parquet"
df = pd.read_parquet(parquet_path)
df['date'] = pd.to_datetime(df['date'])

# Check if 'pet' column exists to be safe
if 'pet' not in df.columns:
    raise ValueError("❌ 'pet' column not found in sarawak_ML_corrected.parquet. Please check your data pipeline.")

print(f"Loaded {len(df)} rows. Stations: {df['station_id'].nunique()}")

# 2. Process ALL stations
all_events_spi = []
all_events_spei = []
all_indices = [] 

for stn_id in df['station_id'].unique():
    stn_data = df[df['station_id'] == stn_id].copy()
    stn_data = stn_data.sort_values('date').set_index('date')
    
    stn_name = stn_data['station_name'].iloc[0] if 'station_name' in stn_data.columns else f"Station_{stn_id}"
    print(f"\n📍 Processing Station {stn_id}: {stn_name}")
    
    # Calculate SPI-12 and SPEI-12 using the existing 'pet' column
    spi_12 = calculate_spi(stn_data['P_corrected'], scale=12)
    spei_12 = calculate_spei(stn_data['P_corrected'], stn_data['pet'], scale=12)
    
    # Save indices for the dashboard
    idx_df = pd.DataFrame({
        'station_id': stn_id,
        'station_name': stn_name,
        'SPI_12': spi_12,
        'SPEI_12': spei_12
    })
    all_indices.append(idx_df)
    
    # Extract Drought Events (Run Theory)
    events_spi = extract_drought_events(spi_12, threshold=-1.0, min_duration=2, inter_event_time=1)
    events_spei = extract_drought_events(spei_12, threshold=-1.0, min_duration=2, inter_event_time=1)
    
    # Calculate Probability/Return Periods and append
    if not events_spi.empty:
        events_spi = calculate_return_periods(events_spi, len(stn_data))
        events_spi['station_id'] = stn_id
        events_spi['station_name'] = stn_name
        events_spi['index_type'] = 'SPI'
        all_events_spi.append(events_spi)
        
    if not events_spei.empty:
        events_spei = calculate_return_periods(events_spei, len(stn_data))
        events_spei['station_id'] = stn_id
        events_spei['station_name'] = stn_name
        events_spei['index_type'] = 'SPEI'
        all_events_spei.append(events_spei)

# 3. Save Outputs
if all_events_spi:
    pd.concat(all_events_spi, ignore_index=True).to_csv(PROJECT_ROOT / "outputs" / "processed" / "drought_events_SPI.csv", index=False)
    print("\n✅ Saved drought_events_SPI.csv")

if all_events_spei:
    pd.concat(all_events_spei, ignore_index=True).to_csv(PROJECT_ROOT / "outputs" / "processed" / "drought_events_SPEI.csv", index=False)
    print("✅ Saved drought_events_SPEI.csv")

if all_indices:
    indices_df = pd.concat(all_indices, ignore_index=True).reset_index().rename(columns={'index': 'date'})
    indices_df.to_parquet(PROJECT_ROOT / "outputs" / "processed" / "station_indices.parquet", index=False)
    print("✅ Saved station_indices.parquet (for dashboard plotting)")

print("\n🎉 All drought extraction complete!")

Loaded 107949 rows. Stations: 273

📍 Processing Station 1003031: Tebedu

📍 Processing Station 1004001: Krusen

📍 Processing Station 1005079: Bukit Matuh

📍 Processing Station 1005080: Sungai Busit

📍 Processing Station 1006028: Sungai Bedup

📍 Processing Station 1006033: Sungai Merang

📍 Processing Station 1006037: Sungai Teb

📍 Processing Station 1007040: Balai Ringin

📍 Processing Station 1008032: Sungai Pinang

📍 Processing Station 1014001: Klauh

📍 Processing Station 1015001: Batu Lintang

📍 Processing Station 1018002: Lupar Antu

📍 Processing Station 1102002: Aman 5 Bridge

📍 Processing Station 1102019: Padawan

📍 Processing Station 1105027: Serian

📍 Processing Station 1105035: Semuja Nonok

📍 Processing Station 1105050: Tebakang

📍 Processing Station 1111008: Pantu

📍 Processing Station 1114019: Temudok

📍 Processing Station 1114028: Melugu

📍 Processing Station 1116001: Engkilili

📍 Processing Station 1118002: Bekatan

📍 Processing Station 1200001: Pang Tebang

📍 Processing Sta